In [158]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

In [146]:
indir = '../DementiaBank-preprocessed2/01-diarized-transcripts-v1/'
files = [f for f in os.listdir(indir) if f.endswith('.csv')]
num_speaker = np.array([len(np.unique(pd.read_csv(os.path.join(indir,file))['speaker'].values)) for file in files])
files = np.array(files)[np.array(num_speaker>1)]
len(files)

258

In [73]:
def df_to_dialogue(df):
    """
    Convert diarized pandas dataframe into readable dialogue text.
    Assumes columns: speaker, text
    """

    # ensure chronological order (important if dataframe was shuffled)
    df = df.sort_values("start_sec")

    lines = []
    for _, row in df.iterrows():
        speaker = row["speaker"]
        #text = row["text"].strip()
        text = row["text"]

        lines.append(f"Speaker {speaker}: {text}")

    dialogue = "\n".join(lines)
    return dialogue

In [42]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

PATIENT_SPEAKER_SCHEMA = {
    "name": "patient_speaker_classification",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "patient_speaker": {"type": "string", "enum": ["A", "B","C","D","E"]},
            "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
            "reason": {"type": "string"}  # keep short; don't include PHI
        },
        "required": ["patient_speaker", "confidence", "reason"]
    }
}

SYSTEM_PROMPT = """You label which speaker (A or B, C, D or E) is the PATIENT in a clinical interview transcript.

Heuristics (use all available evidence, but never output anything except the JSON schema):
- The clinician/examiner typically gives short instructions/questions, changes tasks, and acknowledges ("Okay", "Tell me...", "Are you familiar with...").
- The patient typically produces longer narrative/procedural speech, answers prompts, hesitates, self-corrects, and gives content descriptions.
- If both speak a lot, decide who is driving the tasks vs responding to them.
- You MUST choose even if uncertain; express uncertainty via confidence.
- Do not include any identifying info in the reason. Keep it brief (1–2 sentences).
"""

def classify_patient_speaker(dialogue_text: str, model: str = "gpt-4o-mini") -> dict:
    """
    Returns dict with keys: patient_speaker ("A"|"B"), confidence (0..1), reason (str).
    """

    resp = client.responses.create(
        model=model,
        temperature=0,          # accuracy/repeatability
        top_p=1,
        # max_output_tokens can be small because output is tiny; optional:
        max_output_tokens=200,
        text={
    "format": {
        "type": "json_schema",
        "name": PATIENT_SPEAKER_SCHEMA["name"],
        "schema": PATIENT_SPEAKER_SCHEMA["schema"],
        "strict": True,
    }
},
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": dialogue_text}
        ],
    )

    # Responses API returns structured text; easiest is to parse the first output text as JSON
    out_text = resp.output_text  # should be a JSON string matching schema
    return json.loads(out_text)

In [161]:
var = []
for file in tqdm(files):
    df = pd.read_csv(os.path.join(indir,file))
    dialogue_text = df_to_dialogue(df)
    out = classify_patient_speaker(dialogue_text, model="gpt-4o-mini") 
    var.append(out)

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [06:18<00:00,  1.47s/it]


In [177]:
speakers = pd.DataFrame(var)
speakers['file'] = files
speakers.to_csv('../DementiaBank-preprocessed2/patient_speaker_ids.csv')

In [170]:
# file = files[4]
# df = pd.read_csv(os.path.join(indir,file))
# dialogue_text = df_to_dialogue(df)
# print(dialogue_text)